## 2026 EY AI & Data Challenge - Landsat Data Extraction Notebook

This notebook demonstrates Landsat data extraction and the creation of an output file to be used by the benchmark notebook. The baseline data is [Landsat Collection 2 Level 2](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) data from the MS Planetary Computer catalog.

**Caution**... This notebook requires significant execution time as there are 9,319 data points (unique locations and times) used for data extraction from the Landsat archive. The code takes about 7 hours to run to completion on a typical laptop computer with a typical internet connection. Lower execution times are likely possible with optimization of the data extraction process and the use of cloud computing services.

In [ ]:
# ==== Enhanced feature engineering ====
import numpy as np
import pandas as pd

def compute_band_stats(row, bands=None):
    """
    計算每個波段的統計特徵（均值、標準差、最小值、最大值、四分位數）。
    僅使用每筆資料自身的像素，不跨樣本聚合。
    """
    if bands is None:
        bands = ["blue", "green", "red", "nir", "swir16", "swir22", "st_b10"]
    stats = {}
    for band in bands:
        arr = parse_pixel_array(row.get(band))
        if arr.size == 0:
            stats[f"{band}_mean"] = np.nan
            stats[f"{band}_std"] = np.nan
            stats[f"{band}_min"] = np.nan
            stats[f"{band}_max"] = np.nan
            stats[f"{band}_q25"] = np.nan
            stats[f"{band}_q75"] = np.nan
            stats[f"{band}_count"] = 0
        else:
            stats[f"{band}_mean"] = float(np.nanmean(arr))
            stats[f"{band}_std"] = float(np.nanstd(arr))
            stats[f"{band}_min"] = float(np.nanmin(arr))
            stats[f"{band}_max"] = float(np.nanmax(arr))
            stats[f"{band}_q25"] = float(np.nanquantile(arr, 0.25))
            stats[f"{band}_q75"] = float(np.nanquantile(arr, 0.75))
            stats[f"{band}_count"] = int(np.isfinite(arr).sum())
    return pd.Series(stats)

def compute_index_stats(row, eps=1e-10):
    """
    計算 NDVI、NDMI、MNDWI 指數的統計值（平均、標準差、極值及四分位）。
    僅使用每筆資料自身的像素序列，不做跨樣本聚合。
    """
    b_arr = parse_pixel_array(row.get("blue"))
    g_arr = parse_pixel_array(row.get("green"))
    r_arr = parse_pixel_array(row.get("red"))
    n_arr = parse_pixel_array(row.get("nir"))
    s16_arr = parse_pixel_array(row.get("swir16"))
    def safe_index(x, y):
        m = min(len(x), len(y))
        if m == 0:
            return np.array([], dtype=float)
        x = x[:m]
        y = y[:m]
        return (x - y) / (x + y + eps)
    ndvi = safe_index(n_arr, r_arr)
    ndmi = safe_index(n_arr, s16_arr)
    mndwi = safe_index(g_arr, s16_arr)
    def get_stats(arr, name):
        if arr.size == 0:
            return {
                f"{name}_mean": np.nan,
                f"{name}_std": np.nan,
                f"{name}_min": np.nan,
                f"{name}_max": np.nan,
                f"{name}_q25": np.nan,
                f"{name}_q75": np.nan,
            }
        return {
            f"{name}_mean": float(np.nanmean(arr)),
            f"{name}_std": float(np.nanstd(arr)),
            f"{name}_min": float(np.nanmin(arr)),
            f"{name}_max": float(np.nanmax(arr)),
            f"{name}_q25": float(np.nanquantile(arr, 0.25)),
            f"{name}_q75": float(np.nanquantile(arr, 0.75)),
        }
    stats = {}
    stats.update(get_stats(ndvi, "ndvi"))
    stats.update(get_stats(ndmi, "ndmi"))
    stats.update(get_stats(mndwi, "mndwi"))
    return pd.Series(stats)


### Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process. 

In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os
import time
import re

tqdm.pandas()  

## Raw Data抓取參考

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

def pick_first(keys, asset_keys):
    for k in keys:
        if k in asset_keys:
            return k
    return None

def compute_Landsat_raw(row):
    cols = ["blue", "green", "red", "nir", "swir16", "swir22", "st_b10"]

    try:
        lat, lon = row["Latitude"], row["Longitude"]
        sample_date = pd.to_datetime(row["Sample Date"], dayfirst=True, errors="coerce")
        if pd.isna(sample_date):
            return pd.Series({k: None for k in cols})

        sample_date_utc = (
            sample_date.tz_localize("UTC")
            if sample_date.tzinfo is None
            else sample_date.tz_convert("UTC")
        )

        bbox_size = 0.00089831 * 3
        bbox = [lon - bbox_size/2, lat - bbox_size/2, lon + bbox_size/2, lat + bbox_size/2]

        search = catalog.search(
            collections=["landsat-c2-l2"],
            bbox=bbox,
            datetime="2011-01-01/2015-12-31",
            query={"eo:cloud_cover": {"lt": 10}},
        )
        items = search.item_collection()
        if not items:
            return pd.Series({k: None for k in cols})

        selected_item = min(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties["datetime"]).tz_convert("UTC") - sample_date_utc),
        )

        asset_keys = selected_item.assets.keys()

        nir_key = pick_first(["nir08", "nir"], asset_keys)
        thermal_key = pick_first(["lwir11", "lwir"], asset_keys)

        available_bands = [k for k in ["blue", "green", "red", "swir16", "swir22", nir_key, thermal_key] if k]

        data = stac_load([selected_item], bands=available_bands, bbox=bbox, chunks={}).isel(time=0)

        def get_val(band_name):
            if band_name is None or band_name not in data:
                return None
            return data[band_name].values.flatten().tolist()

        return pd.Series({
            "blue": get_val("blue"),
            "green": get_val("green"),
            "red": get_val("red"),
            "nir": get_val(nir_key),
            "swir16": get_val("swir16"),
            "swir22": get_val("swir22"),
            "st_b10": get_val(thermal_key),  # 2012以前多半會是 ST_B6 對應的 thermal
        })

    except Exception:
        return pd.Series({k: None for k in cols})


def run_landsat_raw_in_batches(df, out_dir, batch_size=100, start_batch=0, pause_every=5, pause_seconds=2):
    os.makedirs(out_dir, exist_ok=True)
    points = df[['Latitude', 'Longitude', 'Sample Date']].drop_duplicates().reset_index(drop=True)
    n = len(points)
    n_batches = (n + batch_size - 1) // batch_size
    
    print(f"Total points: {n}, Batches: {n_batches}")

    for b in range(start_batch, n_batches):
        out_path = f"{out_dir}/landsat_raw_batch_{b:05d}.parquet"
        s, e = b * batch_size, min((b + 1) * batch_size, n)
        
        print(f"🚀 Processing batch {b}/{n_batches-1} (rows {s}..{e-1})")
        
        batch_points = points.iloc[s:e].reset_index(drop=True)
        batch_raw = batch_points.progress_apply(compute_Landsat_raw, axis=1)
        
        pd.concat([batch_points, batch_raw], axis=1).to_parquet(out_path, index=False)
        
        if pause_every and (b - start_batch + 1) % pause_every == 0:
            time.sleep(pause_seconds)

    print("🎉 Done.")

def merge_landsat_batches(out_dir, merged_path):
    files = sorted([os.path.join(out_dir, f) for f in os.listdir(out_dir) if re.match(r"^landsat_raw_batch_\d{5}\.parquet$", f)])
    df_all = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    df_all.to_parquet(merged_path, index=False)
    print(f"✅ Merged {df_all.shape}")
    return df_all

### 用raw檔做各種計算，以median為例

In [ ]:
# --------------------------
# 1) 基礎：把儲存格統一轉成 1D float array
# --------------------------
def parse_pixel_array(val):
    """
    把儲存格內容統一轉成 1D numpy array(float)。
    支援: None / scalar / list / np.ndarray / string like "[1,2,3]"
    """
    if val is None:
        return np.array([], dtype=float)

    # scalar (含 np.nan)
    if np.isscalar(val):
        if pd.isna(val):
            return np.array([], dtype=float)
        return np.array([float(val)], dtype=float)

    # string like "[1,2,3]" or "1 2 3"
    if isinstance(val, str):
        s = val.strip().strip("[]'\"").replace(",", " ")
        s = " ".join(s.split())  # 壓縮多重空白
        if not s:
            return np.array([], dtype=float)
        return np.fromstring(s, sep=" ", dtype=float)

    # list/ndarray
    return np.asarray(val, dtype=float).ravel()


# --------------------------
# 2) 中位數（排除 0）
# --------------------------
def clean_pixel_median(val, zero_as_nan=True):
    """
    處理單一儲存格的像素資料，計算中位數並可選擇排除 0。
    """
    try:
        arr = parse_pixel_array(val)
        if arr.size == 0:
            return np.nan

        med = float(np.nanmedian(arr))
        if zero_as_nan and med == 0:
            return np.nan
        return med
    except Exception:
        return np.nan


# --------------------------
# 3) 光譜指數（用 med_ 波段）
# --------------------------
def calculate_spectral_indices(df, prefix="med_", eps=1e-10):
    """
    根據中位數波段計算遙測指數（in-place 寫回 df）。
    預期欄位：med_blue, med_green, med_red, med_nir, med_swir16
    """
    # 若缺欄位，直接產生 NaN 欄位而不噴錯
    def col(name):
        return df[name] if name in df.columns else np.nan

    b   = col(f"{prefix}blue")
    g   = col(f"{prefix}green")
    r   = col(f"{prefix}red")
    n   = col(f"{prefix}nir")
    s16 = col(f"{prefix}swir16")

    df[f"{prefix}NDMI"]  = (n - s16) / (n + s16 + eps)
    df[f"{prefix}MNDWI"] = (g - s16) / (g + s16 + eps)
    df[f"{prefix}NDVI"]  = (n - r)   / (n + r   + eps)

    df[f"{prefix}NDTI"]  = (r - g)   / (r + g   + eps)
    df[f"{prefix}NDBI"]  = (s16 - n) / (s16 + n + eps)

    df[f"{prefix}UrbanScore"] = df[f"{prefix}NDBI"] - df[f"{prefix}NDVI"]

    # EVI（常見公式）
    df[f"{prefix}EVI"] = 2.5 * (n - r) / (n + 6*r - 7.5*b + 1 + eps)

    return df


# --------------------------
# 4) 安全相關係數
# --------------------------
def safe_corr(a, b):
    """
    計算 corr(a,b)，自動處理長度不同、nan、常數向量等情況。
    """
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()

    m = min(a.size, b.size)
    if m < 2:
        return np.nan

    a = a[:m]
    b = b[:m]

    mask = np.isfinite(a) & np.isfinite(b)
    a = a[mask]
    b = b[mask]
    if a.size < 2:
        return np.nan
    if np.std(a) == 0 or np.std(b) == 0:
        return np.nan

    return float(np.corrcoef(a, b)[0, 1])


# --------------------------
# 5) 像素層級特徵
# --------------------------
def compute_pixel_level_features(
    row,
    eps=1e-10,
    ndvi_thr=0.3,
    ndbi_thr=0.0,
    mndwi_thr=0.0,
    ndti_thr=0.2,
    hot_q=0.75
):
    """
    從 raw_df 的 pixel arrays 計算 ratio / corr / LST stats。
    """

    # --- 取出 arrays ---
    b   = parse_pixel_array(row.get("blue"))
    g   = parse_pixel_array(row.get("green"))
    r   = parse_pixel_array(row.get("red"))
    n   = parse_pixel_array(row.get("nir"))
    s16 = parse_pixel_array(row.get("swir16"))
    lst = parse_pixel_array(row.get("st_b10"))  # LST/熱紅外 band pixel array

    # --- 對齊：用最短長度截斷（修正你的 align_min bug）---
    def align_min(*arrs):
        sizes = [a.size for a in arrs]
        m = min(sizes) if sizes else 0
        if m <= 0:
            return [np.array([], dtype=float) for _ in arrs]
        return [a[:m] for a in arrs]

    # median_LST（把 0 當無效）
    if lst.size:
        median_LST = float(np.nanmedian(lst))
        if median_LST == 0:
            median_LST = np.nan
    else:
        median_LST = np.nan

    # --- pixel-level indices（先對齊再算，避免 broadcast 問題）---
    n_a, r_a = align_min(n, r)
    ndvi = (n_a - r_a) / (n_a + r_a + eps)

    g_a, s16_a = align_min(g, s16)
    mndwi = (g_a - s16_a) / (g_a + s16_a + eps)

    s16_b, n_b = align_min(s16, n)
    ndbi = (s16_b - n_b) / (s16_b + n_b + eps)

    r2_a, g2_a = align_min(r, g)
    ndti = (r2_a - g2_a) / (r2_a + g2_a + eps)

    # --- ratios（布林平均 = 比例）---
    water_ratio = float(np.nanmean(mndwi > mndwi_thr)) if mndwi.size else np.nan
    veg_ratio   = float(np.nanmean(ndvi  > ndvi_thr))  if ndvi.size else np.nan
    bare_ratio  = float(np.nanmean(ndti  > ndti_thr))  if ndti.size else np.nan

    # urban_ratio：需要 ndvi 與 ndbi 同長度
    ndbi_a, ndvi_a = align_min(ndbi, ndvi)
    urban_ratio = (
        float(np.nanmean((ndbi_a > ndbi_thr) & (ndvi_a < ndvi_thr)))
        if ndbi_a.size else np.nan
    )

    # --- hot_ratio：該 row 的 LST 分位數以上比例 ---
    if lst.size:
        q = np.nanquantile(lst, hot_q)
        hot_ratio = float(np.nanmean(lst > q)) if np.isfinite(q) else np.nan
    else:
        hot_ratio = np.nan

    # --- correlations with LST ---
    corr_ndvi_lst  = safe_corr(ndvi,  lst)
    corr_ndbi_lst  = safe_corr(ndbi,  lst)
    corr_mndwi_lst = safe_corr(mndwi, lst)

    return pd.Series({
        "median_LST": median_LST,
        "urban_ratio": urban_ratio,
        "water_ratio": water_ratio,
        "veg_ratio": veg_ratio,
        "bare_ratio": bare_ratio,
        "hot_ratio": hot_ratio,
        "corr_NDVI_LST": corr_ndvi_lst,
        "corr_NDBI_LST": corr_ndbi_lst,
        "corr_MNDWI_LST": corr_mndwi_lst,
    })


# --------------------------
# 6) 主 pipeline
# --------------------------
def process_landsat_pipeline(raw_df, band_cols=None, prefix="med_"):
    """
    主處理流程：
    (1) 計算各波段中位數 -> med_*
    (2) 用 med_* 計算光譜指數
    (3) 用 pixel arrays 計算像素層級特徵
    """
    if band_cols is None:
        band_cols = ["blue", "green", "red", "nir", "swir16", "swir22", "st_b10"]

    # 基本欄位
    base_cols = [c for c in ["Latitude", "Longitude", "Sample Date"] if c in raw_df.columns]
    features_df = raw_df[base_cols].copy()

    print(f"--- 1) 計算各波段中位數 ({len(band_cols)} bands) ---")
    for col in band_cols:
        if col in raw_df.columns:
            features_df[f"{prefix}{col}"] = raw_df[col].apply(clean_pixel_median)
            print(f"   Done: {prefix}{col}")
        else:
            # 欄位不存在也補上，避免後面算指數 KeyError
            features_df[f"{prefix}{col}"] = np.nan
            print(f"   Missing col -> fill NaN: {prefix}{col}")

    print(f"--- 2) 計算光譜指數（用 {prefix} 波段） ---")
    features_df = calculate_spectral_indices(features_df, prefix=prefix)

    print(f"--- 3) 計算像素層級特徵（ratios / corr / hot_ratio） ---")
    pixel_feat_df = raw_df.apply(compute_pixel_level_features, axis=1)
    features_df = pd.concat([features_df, pixel_feat_df], axis=1)

    # 4) 計算波段與指數統計特徵
    band_stats = raw_df.apply(compute_band_stats, axis=1)
    index_stats = raw_df.apply(compute_index_stats, axis=1)
    features_df = pd.concat([features_df, band_stats, index_stats], axis=1)

    # 5) 加入時間相關特徵
    if "Sample Date" in features_df.columns:
        dt = pd.to_datetime(features_df["Sample Date"], dayfirst=True, errors="coerce")
        features_df["month"] = dt.dt.month
        features_df["dayofyear"] = dt.dt.dayofyear

    print("--- 處理完成 ---")
    return features_df

In [ ]:
# ==========================================
# 實際執行 (Main Execution)
# ==========================================

# 1. 載入資料
path = "landsat_raw_all_v3.parquet"
landsat_raw = pd.read_parquet(path)


# 2. 執行 Pipeline
landsat_train_features = process_landsat_pipeline(landsat_raw)

# 3. 儲存結果
output_path = "/tmp/landsat_features_training_200m_v2.csv"
landsat_train_features.to_csv(output_path, index=False)
print(f"結果已儲存至: {output_path}")

# 7) 上傳至 Snowflake Stage
session.sql(f"""
    PUT file://{output_path}
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved and uploaded! Refresh the browser to see the files in the sidebar.")

# 預覽結果
landsat_train_features.head()

**NDMI and MNDWI Indices**

In this notebook, we compute two commonly used water-related indices from the extracted Landsat bands:

- **NDMI (Normalized Difference Moisture Index):**  
  Measures vegetation water content and surface moisture.  
  Computed as *(NIR - SWIR16) / (NIR + SWIR16)*.

- **MNDWI (Modified Normalized Difference Water Index):**  
  Highlights open water features by enhancing water reflectance and suppressing built-up areas.  
  Computed as *(Green - SWIR16) / (Green + SWIR16)*.

An **epsilon value** (*eps = 1e-10*) is added to the denominators to avoid division by zero.  
These indices are widely used in hydrological and water quality analyses for detecting water presence and vegetation moisture levels.

**Note:** If you're using your own workspace, remember to replace "EY-AI-and-Data-Challenge" with your workspace name in the file path.

### Extracting features for the validation dataset

In [11]:
Validation_df=pd.read_csv('submission_template.csv')
display(Validation_df.head())

In [12]:
Validation_df.shape

In [ ]:
# -----------------------------
# 執行
# -----------------------------
run_landsat_raw_in_batches(
        df=Validation_df,
        out_dir="landsat_validation_raw_batches",
        batch_size=100,
        start_batch=0 
    )
    
merge_landsat_batches(
        out_dir="landsat_validation_raw_batches",
        merged_path="landsat_validation_raw_all_v3.parquet"
    )

In [ ]:
merge_landsat_batches(
        out_dir="landsat_validation_raw_batches",
        merged_path="/tmp/landsat_validation_raw_all_v3.parquet"
    )
session.sql("""
    PUT file:///tmp/landsat_validation_raw_all_v3.parquet
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")

In [ ]:
# 1. 載入資料
landsat_validation_raw = pd.read_parquet("landsat_validation_raw_all_v3.parquet")

# 2. 執行 Pipeline
landsat_validation_features = process_landsat_pipeline(landsat_validation_raw)

# 3. 儲存結果
output_path = "/tmp/landsat_features_validation_200m_v2.csv"
landsat_validation_features.to_csv(output_path, index=False)
print(f"結果已儲存至: {output_path}")

# 7) 上傳至 Snowflake Stage
session.sql(f"""
    PUT file://{output_path}
    'snow://workspace/USER$.PUBLIC."EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved and uploaded! Refresh the browser to see the files in the sidebar.")

# 預覽結果
landsat_validation_features.head()

**Note:** If you're using your own workspace, remember to replace "EY-AI-and-Data-Challenge" with your workspace name in the file path.